In [6]:
from astropy.time import Time
from lsst.summit.utils.efdUtils import makeEfdClient
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


from collections import defaultdict
from datetime import datetime, timedelta
import numpy as np
import pandas as pd

from lsst.ts.xml.tables.m1m3 import FATable

#%matplotlib inline
#%load_ext lab_black

import asyncio
import ipywidgets as widgets
from IPython.display import display, clear_output

import nest_asyncio
nest_asyncio.apply()

<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>: (This is the expected behaviour if you are running under QEMU)


# M1M3 Bump Test Log Error Analysis and Measured Forces
---

## Overview
This notebook is dedicated to the analysis of bump test log error data, which is stored under the `lsst.sal.MTM1M3.logevent_logMessage` EFD topic. Our primary focus is on messages that include the "*Failed FA*" string, as this string is generated each time a bump test fails.

The returned messages typically contain a string similar to this: "*measured force plus (215.394) is too far 222±5*". Our goal is to extract the measured forces (enclosed in parentheses) and analyze the severity of actuator failures by comparing these measured forces with the expected forces.

## Helper Functinos

### Querying and Processing Scrit Queue Executions}

In [68]:
async def query_script_queue_logs(past_hours=12, client_name="usdf_efd"):
    """
    Queries the log messages related to the script queue from the EFD.

    Parameters:
    ------------
    past_hours (int, optional): Number of hours to query. Defaults to 12.

    client_name (str, optional): Name of the EFD client. Defaults to "usdf_efd".

    Returns:
    ------------
    script_logs (pandas.core.frame.DataFrame): List of log messages related to the script
    """

    # Get current time
    now = datetime.now()
    end = Time(now.isoformat(), format="isot", scale="utc")
    start = Time((now - timedelta(hours=past_hours)).isoformat(), format="isot", scale="utc")

    # Create the EFD client
    possible_clients = ["summit_efd", "usdf_efd"]

    if client_name not in possible_clients:
        print(f"Invalid client name. Possible clients: {possible_clients}")
        return None

    client = makeEfdClient(client_name)
    script_logs = await client.select_time_series(
        topic_name="lsst.sal.ScriptQueue.logevent_script",
        fields="*",
        start=start,
        end=end,
    )

    return script_logs

def filter_and_process_queue_logs(script_logs):
    """
    Filters and processes the script logs DataFrame.

    Parameters:
    ------------
    script_logs (DataFrame): DataFrame containing the script logs.

    Returns:
    ------------
    DataFrame: Processed DataFrame.
    """

    processState_mapping = {
        0: "UNKNOWN",
        1: "LOADING",
        2: "CONFIGURED",
        3: "RUNNING",
        4: "DONE",
        5: "LOADFAILED",
        6: "CONFIGURE_FAILED",
        7: "TERMINATED",
    }

    scriptState_mapping = {
        0: "UNKNOWN",
        1: "UNCONFIGURED",
        2: "CONFIGURED",
        3: "RUNNING",
        4: "PAUSED",
        5: "ENDING",
        6: "STOPPING",
        7: "FAILING",
        8: "DONE",
        9: "STOPPED",
        10: "FAILED",
        11: "CONFIGURE_FAILED",
    }

    # Script path and salIndex
    path = "maintel/m1m3/check_actuators.py"
    salIndex = 1

    df_script_logs = pd.DataFrame(script_logs)

    # Filter script_logs for path ==  "maintel/m1m3/check_hardpoint.py" and salIndex == 1
    df_filtered_logs = df_script_logs[(df_script_logs["path"] == path) & (df_script_logs["salIndex"] == salIndex)]

    # Get timestamps
    df_filtered_logs = df_filtered_logs.set_index("private_rcvStamp")
    df_filtered_logs.index = pd.to_datetime(df_filtered_logs.index, unit="s")

    # Map the processState and scriptState values to their corresponding strings
    df_filtered_logs["processState_str"] = df_filtered_logs["processState"].map(processState_mapping)
    df_filtered_logs["scriptState_str"] = df_filtered_logs["scriptState"].map(scriptState_mapping)

    # Columns to remove
    columns_to_remove = [
        "blockId",
        "blockSize",
        "cmdId",
        "executionId",
        "private_efdStamp",
        "private_kafkaStamp",
        "private_origin",
        "private_revCode",
        "private_sndStamp",
        "private_seqNum",
        "scriptBlockIndex",
        "isStandard",
        "private_identity",
        "processState",
        "scriptState"
    ]

    # Remove the specified columns
    df_filtered_logs.drop(columns=columns_to_remove, inplace=True)

    # Sort from most recent events to oldest
    df_filtered_logs.sort_index(ascending=False, inplace=True)

    return df_filtered_logs


def extract_execution_details(df_check_actuators_log):
    """
    Extracts execution details from the script logs DataFrame.

    Parameters:
    df_check_actuators_log (DataFrame): DataFrame containing the script logs.

    Returns:
    DataFrame: DataFrame containing execution details.
    """
    # Initialize an empty list to store execution details
    execution_data = []

    # Convert all columns starting with 'timestamp' to datetime
    timestamp_columns = [col for col in script_logs_processed.columns if col.startswith('timestamp')]
    for col in timestamp_columns:
        script_logs_processed[f'{col}'] = pd.to_datetime(script_logs_processed[col], unit="s")

    # Loop over each unique scriptSalIndex
    for script_sal_index in df_check_actuators_log['scriptSalIndex'].unique():
        df_sal = df_check_actuators_log[df_check_actuators_log['scriptSalIndex'] == script_sal_index]
        df_sal = df_sal.sort_index()  # Ensure chronological order

        # Initialize start_time for each script_sal_index
        start_time = None

        # Iterate over the rows to find start and end times for each execution
        for idx, row in df_sal.iterrows():
            if row['processState_str'] == 'CONFIGURED' and start_time is None:
                start_time = row['timestampConfigureStart']  # Mark the start time
            elif row['processState_str'] == 'DONE' and start_time is not None:
                end_time = row['timestampProcessEnd']  # Mark the end time
                execution_data.append({
                    'scriptSalIndex': script_sal_index,
                    'start_time': start_time,
                    'end_time': end_time,
                })
                start_time = None  # Reset for the next execution

    # Create the DataFrame
    df_executions = pd.DataFrame(execution_data)

    if not df_executions.empty:
        # Calculate duration in minutes
        df_executions['duration_minutes'] = (
            df_executions['end_time'] - df_executions['start_time']
        ).dt.total_seconds() / 60.

        # Reorder columns like this
        cols = ['scriptSalIndex', 'start_time', 'end_time', 'duration_minutes']
        df_executions = df_executions[cols]

        # Sort by start time in descending order
        df_executions = df_executions.sort_values('start_time', ascending=False)
    else:
        print("No executions found.")

    return df_executions



### Querying and Processing Bump Test Log Messsages

In [64]:
async def query_bump_logs(start_date, end_date, client_name="summit_efd"):
    """
    Queries the log messages related to bump tests from the EFD.

    Parameters:
    ------------
    start_date (str): Start date of the query in ISO format (YYYY-MM-DD).

    end_date (str): End date of the query in ISO format (YYYY-MM-DD).

    client_name (str, optional): Name of the EFD client. Defaults to "summit_efd".

    Returns:
    ------------
    bump_logs (pandas.core.frame.DataFrame): List of log messages related to bump tests
    """

    # Convert start and end dates to datetime objects
    start = datetime.fromisoformat(start_date)
    end = datetime.fromisoformat(end_date)

    # Create the EFD client
    possible_clients = ["summit_efd", "usdf_efd"]
    if client_name not in possible_clients:
        print(f"Invalid client name. Possible clients: {possible_clients}")
        return None

    client = makeEfdClient(client_name)

    try:
        bump_logs = await client.select_time_series(
                    topic_name="lsst.sal.MTM1M3.logevent_logMessage",
                    fields=["message"],
                    start=Time(start.isoformat(), format="isot", scale="utc"),
                    end=Time(end.isoformat(), format="isot", scale="utc"),
            )
    except Exception as e:
        print(
            f"Error querying data from {start.isoformat()} to {end.isoformat()}: {e}"
        )
    return bump_logs


# Function to process the bump log
def process_bump_logs(bump_logs, expected_force_range=222, tolerance=5):
    """
    Processes bump log messages to extract relevant information and calculate deviations
    from expected forces.

    Parameters:
    bump_logs (DataFrame): DataFrame containing bump log messages.

    Returns:
    DataFrame: A processed DataFrame with extracted and calculated data.
    """
    # Filter the bump logs
    filtered_bump_log = bump_logs[bump_logs.message.str.contains("Failed FA")].message
    df_filtered_bump_log = pd.DataFrame(filtered_bump_log)

    # Process the bump log messages
    # Extract FA ID, Orientation, Index, and Error Message
    df_filtered_bump_log["ID"] = df_filtered_bump_log["message"].str.extract(
        r"FA ID (\d+)"
    )
    orientation_index = df_filtered_bump_log["message"].str.extract(r"\(([XYZ])(\d+)\)")
    df_filtered_bump_log["Orientation"] = orientation_index[0]
    df_filtered_bump_log["Index"] = orientation_index[1]
    df_filtered_bump_log["Error Message"] = df_filtered_bump_log["message"].str.extract(
        r"- (.+)$"
    )

    # Create a new DataFrame with the required columns
    new_df = df_filtered_bump_log.reset_index().rename(columns={"index": "Time"})[
        ["Time", "ID", "Orientation", "Index", "Error Message"]
    ]

    # Calculate deviations and classify error messages
    # Extract Measured Force
    new_df["MeasuredForce"] = (
        new_df["Error Message"].str.extract(r"\(([\-\d.]+)\)").astype(float)
    )

    # Filter out entries with zero force applied - to remove disabled actuators
    new_df = new_df[new_df["MeasuredForce"] != 0]

    # Classify applied force direction and calculate deviation
    new_df["AppliedForceDirection"] = new_df["Error Message"].apply(
        lambda x: "Positive" if "measured force plus" in x else "Negative"
    )

    # Initialize deviation columns
    new_df["Deviation"] = 0

    # Upper and lower limits are equal to the expected force range (+/- tolerance is optional if needed)
    upper_limit = expected_force_range  # + tolerance
    lower_limit = -expected_force_range  # - tolerance

    # Ensure Deviation column is float
    new_df["Deviation"] = new_df["Deviation"].astype(float)

    # Calculate deviation based on the direction of the applied force direction
    for index, row in new_df.iterrows():
        if row["AppliedForceDirection"] == "Positive":
            new_df.at[index, "Deviation"] = float(row["MeasuredForce"]) - upper_limit
        elif row["AppliedForceDirection"] == "Negative":
            new_df.at[index, "Deviation"] = float(row["MeasuredForce"]) - lower_limit

    cols = ["Time", "ID", "Orientation", "Index", "Error Message", "AppliedForceDirection",  "MeasuredForce", "Deviation"]

    return new_df[cols]


### Display Functions

In [69]:
def plot_measured_forces_with_deviation(df):
    """
    Function to plot the measured forces by FA ID, differentiating between:
    - Applied force direction (Positive or Negative)
    - Deviation type (Overshoot or Undershoot)

    Parameters:
    df (DataFrame): DataFrame containing the processed bump log data.
    """
    if df.empty:
        print("The DataFrame is empty. No failure to show.")
        return

    # Expected force range and tolerance
    expected_force = 222
    tolerance = 5

    # Create a new column for deviation type
    def classify_deviation(row):
        deviation = abs(row['MeasuredForce']) - expected_force
        if deviation > 0:
            return 'Overshoot'
        else:
            return 'Undershoot'

    df['DeviationType'] = df.apply(classify_deviation, axis=1)

    # Sort the IDs for consistent plotting
    df['ID'] = df['ID'].astype(str)
    sorted_ids = df['ID'].sort_values().unique()

    # Create color and marker mapping for DeviationType
    deviation_styles = {
        'Overshoot': {'color': 'red', 'marker': 'o'},
        'Undershoot': {'color': 'green', 'marker': 's'}
    }

    # Create figure and axis
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

    # Map actuator IDs to positions on x-axis
    id_to_position = {id_: pos for pos, id_ in enumerate(sorted_ids)}
    df['ID_Pos'] = df['ID'].map(id_to_position)

    # Scatter plot for Positive AppliedForceDirection
    for dev_type, group in df[df['AppliedForceDirection'] == 'Positive'].groupby('DeviationType'):
        ax1.scatter(
            group['ID_Pos'],
            group['MeasuredForce'],
            c=deviation_styles[dev_type]['color'],
            marker=deviation_styles[dev_type]['marker'],
            label=f'Positive - {dev_type}',
            alpha=0.7,
        )

    # Scatter plot for Negative AppliedForceDirection
    for dev_type, group in df[df['AppliedForceDirection'] == 'Negative'].groupby('DeviationType'):
        ax2.scatter(
            group['ID_Pos'],
            group['MeasuredForce'],
            c=deviation_styles[dev_type]['color'],
            marker=deviation_styles[dev_type]['marker'],
            label=f'Negative - {dev_type}',
            alpha=0.7,
        )

    # Adding horizontal lines for expected force and tolerance
    ax1.axhline(expected_force, color='gray', linestyle='--', linewidth=0.8)
    ax1.fill_between(
        [-1, len(sorted_ids)],
        expected_force - tolerance,
        expected_force + tolerance,
        color='gray',
        alpha=0.2,
        label='Tolerance Range',
    )

    ax2.axhline(-expected_force, color='gray', linestyle='--', linewidth=0.8)
    ax2.fill_between(
        [-1, len(sorted_ids)],
        -expected_force - tolerance,
        -expected_force + tolerance,
        color='gray',
        alpha=0.2,
    )

    # Set x-ticks and labels
    ax2.set_xticks(range(len(sorted_ids)))
    ax2.set_xticklabels(sorted_ids, rotation=45, ha='center')

    # Set y-axis limits for each subplot
    y_max1 = df[df['AppliedForceDirection'] == 'Positive']['MeasuredForce'].max() + tolerance + 25
    y_min1 = df[df['AppliedForceDirection'] == 'Positive']['MeasuredForce'].min() - tolerance - 25
    y_max2 = df[df['AppliedForceDirection'] == 'Negative']['MeasuredForce'].max() + tolerance + 25
    y_min2 = df[df['AppliedForceDirection'] == 'Negative']['MeasuredForce'].min() - tolerance - 25
    # Dynamically adjust y-axis limits based on the data

    ax1.set_ylim(y_min1, y_max1)
    ax2.set_ylim(y_min2, y_max2)

    # Set labels and title
    ax2.set_xlabel('FA ID')
    ax1.set_ylabel('Measured Force (N)')
    ax2.set_ylabel('Measured Force (N)')

    # Set title inside each subplot
    ax1.set_title('Measured Forces by FA ID with Deviation Classification (Positive)')
    ax2.set_title('Measured Forces by FA ID with Deviation Classification (Negative)')

    # Create custom legends
    deviation_legend_elements = [
        Line2D([0], [0], marker=style['marker'], color='w', label=dev_type,
               markerfacecolor=style['color'], markersize=8)
        for dev_type, style in deviation_styles.items()
    ]

    ax1.legend(handles=deviation_legend_elements, title='Deviation Type', loc='upper left')

    plt.tight_layout()
    display(fig)


# Getting primary {id: index} dictionary
m1m3_actuator_id_index_table = {fa.actuator_id: fa.index for fa in FATable}

def get_m1m3_actuator_ids():
    """Get a list of the M1M3 actuator ids."""
    return list(m1m3_actuator_id_index_table.keys())

def get_xy_position(actuator_list=FATable):
    xpos = [actuator.x_position for actuator in actuator_list]
    ypos = [actuator.y_position for actuator in actuator_list]
    return xpos, ypos

def plot_failure_histogram(df, ax):
    """
    Function to plot a histogram of the deviations from the expected force range.

    Parameters:
    ------------
    df (DataFrame): DataFrame containing the processed bump log data.
    ax (matplotlib.axes.Axes): Axes object to plot the histogram.
    """

    # Define color scheme for orientations
    orientation_colors = {"X": "blue", "Y": "red", "Z": "green"}

    # Ensure 'ID' and 'Orientation' are categorical for the observed parameter to have an effect
    df["ID"] = df["ID"].astype('category')
    df["Orientation"] = df["Orientation"].astype('category')

    # Aggregate the total number of failures per FA ID
    total_failures_per_faid = df.groupby("ID", observed=True).size()

    # Group by 'ID' and 'Orientation', specifying observed=True
    grouped_data = df.groupby(["ID", "Orientation"], observed=True).size().unstack(fill_value=0)

    # Sort the grouped data based on total failures
    sorted_grouped_data = grouped_data.reindex(
        total_failures_per_faid.sort_values(ascending=False).index
    )

    # Colors for present orientations
    bar_colors = {
        ori: orientation_colors.get(ori, 'gray')
        for ori in grouped_data.columns
    }

    # Plotting with specified colors
    sorted_grouped_data.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        alpha=0.8,
        color=[bar_colors.get(col, 'gray') for col in grouped_data.columns],
    )

    # Set title and labels
    ax.set_title("Failures by FA ID and Orientation")
    ax.set_xlabel("FA ID")
    ax.set_ylabel("Number of Failures")

    ax.legend(title="Orientation", frameon=False)

def ActuatorsLayout(ax, df, actuator_list=FATable):
    """
    Function to plot the layout of M1M3 actuators and highlight failed actuators.

    Parameters:
    ax (matplotlib.axes.Axes): Axes object to plot the actuators.
    df (DataFrame): DataFrame containing failure data including 'ID' and 'Orientation'.
    actuator_list (list): List of actuator objects, default is FATable.
    """
    # Define color schemes for orientations
    orientation_colors = {"X": "blue", "Y": "red", "Z": "green"}
    combined_orientations_colors = {"XZ": "orange", "YZ": "black"}

    # Set axis labels and title
    ax.set_xlabel("X position (m)")
    ax.set_ylabel("Y position (m)")
    ax.set_title("Failures Distribution", fontsize=12)

    # Get actuator IDs and their positions
    ids = get_m1m3_actuator_ids()
    xpos, ypos = get_xy_position(actuator_list)

    # Create a dictionary to store orientations for each actuator
    actuator_orientations = defaultdict(set)
    for actuator_id, orientation in zip(df["ID"], df["Orientation"]):
        actuator_orientations[int(actuator_id)].add(orientation)

    # Separate unique and combined orientations
    unique_orientations = {
        ori
        for orientations in actuator_orientations.values()
        if len(orientations) == 1
        for ori in orientations
    }
    combined_orientations = {
        "".join(sorted(orientations))
        for orientations in actuator_orientations.values()
        if len(orientations) > 1
    }

    # Plot all actuators with low opacity
    ax.plot(xpos, ypos, "o", ms=14, color="blue", alpha=0.05, mec="red")
    for l, x, y in zip(ids, xpos, ypos):
        ax.annotate(
            l, (x, y), textcoords="offset points", xytext=(-5.5, -2), color="blue", size="xx-small"
        )

    # Highlight failed actuators with specific colors
    for actuator_id, orientations in actuator_orientations.items():
        if actuator_id in m1m3_actuator_id_index_table:
            index = m1m3_actuator_id_index_table[actuator_id]
            color = (
                combined_orientations_colors["".join(sorted(orientations))]
                if len(orientations) > 1
                else orientation_colors[orientations.pop()]
            )
            ax.scatter(
                xpos[index],
                ypos[index],
                marker="o",
                facecolors="none",
                edgecolors=color,
                s=250,
                alpha=0.5,
                linewidths=2,
            )

    # Add legend for unique orientations
    for orientation in unique_orientations:
        ax.scatter(
            [],
            [],
            marker="o",
            linestyle="None",
            s=10,
            facecolor="none",
            edgecolor=orientation_colors[orientation],
            alpha=0.9,
            label=f"Orientation {orientation}",
        )
    # Add legend for combined orientations
    for combined_orientation in combined_orientations:
        ax.scatter(
            [],
            [],
            marker="o",
            linestyle="None",
            s=10,
            facecolor="none",
            edgecolor=combined_orientations_colors[combined_orientation],
            alpha=0.5,
            label=f"Orientation {combined_orientation}",
        )
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1))


def plot_histogram_and_layout(df, FATable):
    """
    Function to plot both the histogram of failures and the layout of actuators.

    Parameters:
    df (DataFrame): DataFrame containing the processed bump log data.
    FATable (list): List of actuator objects.
    """
    if df.empty:
        print("The DataFrame is empty. No failure to show.")
        return

    fig, [ax0, ax1] = plt.subplots(1, 2, figsize=(13, 5))
    ax0.set_aspect("auto", adjustable="box")
    ax1.set_aspect("equal", adjustable="box")

    # Plot the failure histogram
    plot_failure_histogram(df, ax0)
    # Plot the actuators layout
    ActuatorsLayout(ax1, df, actuator_list=FATable)

    # Set the overall title with the date range
    first_date = df["Time"].min()
    last_date = df["Time"].max()
    plt.suptitle(
        f"Start: {first_date.strftime('%Y-%m-%d %H:%M:%S')}   End: {last_date.strftime('%Y-%m-%d %H:%M:%S')}"
    )
    plt.tight_layout()
    display(fig)


# Results
---

### Show Executions

In [70]:
script_logs = await query_script_queue_logs(past_hours=6, client_name="summit_efd")

script_logs_processed = filter_and_process_queue_logs(script_logs)

df_executions = extract_execution_details(script_logs_processed)

# Filter script_logs_processed to keep only  scriptSalIndex = 102128
filtered_script_logs_processed = script_logs_processed[script_logs_processed['scriptSalIndex'] == 102128]

In [71]:
df_executions

,scriptSalIndex,start_time,end_time,duration_minutes
0,102133,2024-12-10 21:42:26.497469664,2024-12-10 21:42:40.770392179,0.237882
1,102129,2024-12-10 21:22:56.611423016,2024-12-10 21:23:13.789692879,0.286304
2,102128,2024-12-10 20:11:51.732022524,2024-12-10 21:17:26.701204777,65.582820


In [72]:
filtered_script_logs_processed

,path,salIndex,scriptSalIndex,timestampConfigureEnd,timestampConfigureStart,timestampProcessEnd,timestampProcessStart,timestampRunStart,processState_str,scriptState_str
private_rcvStamp,,,,,,,,,,
2024-12-10 21:17:26.701730967,maintel/m1m3/check_actuators.py,1,102128,2024-12-10 20:11:54.072618484,2024-12-10 20:11:51.732022524,2024-12-10 21:17:26.701204777,2024-12-10 20:11:44.505693674,2024-12-10 20:11:54.076745987,DONE,FAILED
2024-12-10 21:17:24.551158428,maintel/m1m3/check_actuators.py,1,102128,2024-12-10 20:11:54.072618484,2024-12-10 20:11:51.732022524,1970-01-01 00:00:00.000000000,2024-12-10 20:11:44.505693674,2024-12-10 20:11:54.076745987,RUNNING,FAILED
2024-12-10 21:17:24.534128428,maintel/m1m3/check_actuators.py,1,102128,2024-12-10 20:11:54.072618484,2024-12-10 20:11:51.732022524,1970-01-01 00:00:00.000000000,2024-12-10 20:11:44.505693674,2024-12-10 20:11:54.076745987,RUNNING,FAILING
2024-12-10 20:11:54.080439091,maintel/m1m3/check_actuators.py,1,102128,2024-12-10 20:11:54.072618484,2024-12-10 20:11:51.732022524,1970-01-01 00:00:00.000000000,2024-12-10 20:11:44.505693674,2024-12-10 20:11:54.076745987,RUNNING,RUNNING
2024-12-10 20:11:54.072963238,maintel/m1m3/check_actuators.py,1,102128,2024-12-10 20:11:54.072618484,2024-12-10 20:11:51.732022524,1970-01-01 00:00:00.000000000,2024-12-10 20:11:44.505693674,1970-01-01 00:00:00.000000000,CONFIGURED,CONFIGURED
2024-12-10 20:11:54.070927620,maintel/m1m3/check_actuators.py,1,102128,1970-01-01 00:00:00.000000000,2024-12-10 20:11:51.732022524,1970-01-01 00:00:00.000000000,2024-12-10 20:11:44.505693674,1970-01-01 00:00:00.000000000,LOADING,CONFIGURED
2024-12-10 20:11:51.766483545,maintel/m1m3/check_actuators.py,1,102128,1970-01-01 00:00:00.000000000,2024-12-10 20:11:51.732022524,1970-01-01 00:00:00.000000000,2024-12-10 20:11:44.505693674,1970-01-01 00:00:00.000000000,LOADING,UNCONFIGURED
2024-12-10 20:11:44.505954981,maintel/m1m3/check_actuators.py,1,102128,1970-01-01 00:00:00.000000000,1970-01-01 00:00:00.000000000,1970-01-01 00:00:00.000000000,2024-12-10 20:11:44.505693674,1970-01-01 00:00:00.000000000,LOADING,UNKNOWN


In [ ]:
%matplotlib notebook


# Create widgets
script_sal_index_selector = widgets.Dropdown(
    options=df_executions['scriptSalIndex'].unique(),
    value=df_executions['scriptSalIndex'].iloc[-1],
    description='SalIndex:',
    disabled=False,
)

client_selector = widgets.Dropdown(
    options=["summit_efd", "usdf_efd"],
    value="usdf_efd",
    description='EFD Client:',
    disabled=False,
)

query_button = widgets.Button(
    description='Get FA results',
    disabled=False,
    button_style='info',
    tooltip='Click to query/process bump logs',
    icon='search'
)

# Define the UI
ui = widgets.VBox([script_sal_index_selector, client_selector, query_button])


# Define the event handler
async def on_query_button_clicked(b):

    # Clear previous outputs
    clear_output(wait=True)
    display(ui)  # Re-display the UI after clearing outputs

    selected_script_sal_index = script_sal_index_selector.value
    execution_row = df_executions[df_executions['scriptSalIndex'] == selected_script_sal_index].iloc[0]
    t_start = execution_row['start_time']
    t_end = execution_row['end_time']
    client_name = client_selector.value

    print(
          f"Querying bump logs for scriptSalIndex {selected_script_sal_index} "
          f"from {t_start} to {t_end}...\n"
    )

    # Convert times to Time objects
    start_time = Time(t_start)
    end_time = Time(t_end)

    # Call the query function
    try:
        bump_logs = await query_bump_logs(start_time.iso, end_time.iso, client_name=client_name)

        # If bump logs are available, process and display them
        if bump_logs is not None and not bump_logs.empty:
            # Process the bump logs
            processed_bump_logs = process_bump_logs(bump_logs)
            print("\nBump Test Log Messages: ")
            display(processed_bump_logs)

            # Generate and display the plot
            try:
                print("\nMeasured Forces:")
                plot_measured_forces_with_deviation(processed_bump_logs)

                print("\nHistogram and Failures Distribution:")
                plot_histogram_and_layout(processed_bump_logs, FATable)

            except Exception as e:
                print(f"An error occurred while plotting: {e}")
        else:
            print("No bump logs found for the selected execution.")
    except Exception as e:
        print(f"An error occurred while querying bump logs: {e}")

# Attach the event handler
query_button.on_click(lambda b: asyncio.ensure_future(on_query_button_clicked(b)))

# Display the widgets
display(ui)

Querying bump logs for scriptSalIndex 102129 from 2024-12-10 21:22:56.611423016 to 2024-12-10 21:23:13.789692879...

No bump logs found for the selected execution.


## Processing Bump Log

Here we will process the log messages to extract the measured forces and return a dataframe with the measured forces and the expected forces.

### Helper Functions

## Histogram + Spatial distribution

Here, we first process the log messages to extract the measured forces and return a dataframe with the measured forces, direction of the force measured, deviations, etc. Then, we use the helper function defined on sections above to create a bar plot of the failures. In order to see the spatial distribution of the actuators that failed the bump test, we also plot the spatial distribution of the failed actuators indicating the direction of the failed actuators.

## Measured Forces vs Expected Forces

We can also use the processed dataframe to plot the measured forces and the deviations vs the expected forces. Below, we plot the absolute measured forces for each actuator, either for forces applied in the positive or negative direction. The plot includes an color bar, which may help to identify how old the failures are and if the measured forces get worse over time (which we found is not particularly true).